# qwen_vl_model

Final source: `final-submit`. Scoring: Micro-F1. All run flags are off until explicitly enabled.


## 1. Environment and configuration

Kaggle: attach `vhngthnh/reconia-data` and provide `GH_TOKEN` for the private repository.

Qwen requires CUDA, bitsandbytes and PEFT. Install requirements-qwen.txt explicitly in a GPU environment. The linked published result is WildReceipt only; no job is launched by this notebook.


In [ ]:
MODEL = "qwen_vl"
RUN_PREPARE = False
RUN_DOWNLOAD = False
RUN_TRAIN = False
RUN_EXPORT = False
TRAIN_CONFIG = {'epochs': 25, 'batch_size': 2, 'accum_steps': 4, 'lr': 0.0001, 'patience': 5, 'weight_decay': 0.01, 'warmup_ratio': 0.1}

import os
import sys
import subprocess
from pathlib import Path

DATASET = "wildreceipt"
GIT_REF = "final-submit"
repo_dir = Path("/kaggle/working/reconia") if Path("/kaggle").exists() else Path.cwd()
if repo_dir.name == "notebooks":
    repo_dir = repo_dir.parent
if not repo_dir.exists():
    import base64
    from kaggle_secrets import UserSecretsClient
    env = os.environ.copy()
    token = UserSecretsClient().get_secret("GH_TOKEN")
    auth = base64.b64encode(("x-access-token:" + token).encode()).decode()
    env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
               GIT_CONFIG_VALUE_0="Authorization: Basic " + auth)
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF,
                    "https://github.com/tamir39/reconia.git", str(repo_dir)], env=env, check=True)
    del env, token, auth
if (repo_dir / ".git").exists():
    source_branch = subprocess.check_output(
        ["git", "-C", str(repo_dir), "branch", "--show-current"], text=True
    ).strip()
    if source_branch != GIT_REF:
        raise RuntimeError(
            f"Expected branch {GIT_REF}, found {source_branch or 'detached HEAD'}. "
            "Use a fresh Kaggle session or update the checkout explicitly before running."
        )
    source_commit = subprocess.check_output(
        ["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True
    ).strip()
    print("Source:", source_branch, source_commit)
elif (repo_dir / "demo/artifacts.json").is_file():
    print("Source: exported final-submit package")
else:
    raise RuntimeError("Open this notebook from the Reconia repository or final source package")
os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))
if RUN_TRAIN:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-qwen.txt"], check=True)
else:
    print("Qwen GPU dependency install disabled with RUN_TRAIN=False.")


## 2. Prepare and verify data

Reuse the published splits or enable preparation from raw data. The manifest gate rejects changed OCR, labels or split order.


In [ ]:
from importlib import import_module
from tools.verify_final import verify_prepared
from utils.common import get_cache_dir

datasets = ("sroie", "cord_v2", "wildreceipt") if DATASET == "all" else (DATASET,)
if RUN_PREPARE:
    for dataset in datasets:
        getattr(import_module(f"prepare.prepare_{dataset}"), f"prepare_{dataset}")()
if RUN_TRAIN:
    verify_prepared(datasets)
else:
    print("Preparation/training disabled. Existing files are kept.")


## 3. Pretrained model


In [ ]:
if RUN_DOWNLOAD and MODEL != "pick":
    from tools.download_weights import download_weights
    download_weights(MODEL)
elif RUN_DOWNLOAD:
    print("PICK loads its pretrained ResNet through Torchvision when training.")


## 4. Train, validate and evaluate best checkpoint

The published trainer selects validation F1 and evaluates test after training. No training is performed unless RUN_TRAIN is enabled.


In [ ]:
if RUN_TRAIN:
    trainer = import_module(f"models.{MODEL}.train")
    for dataset in datasets:
        trainer.train_model(dataset, **TRAIN_CONFIG, resume=False)
else:
    print("RUN_TRAIN=False; no training started.")


## 5. Inspect output


In [ ]:
import json
for dataset in datasets:
    result = repo_dir / "results" / dataset / MODEL
    metrics = result / "test_metrics.json"
    print(dataset, json.loads(metrics.read_text()) if metrics.exists() else "No local evaluation output yet")


## 6. Export checkpoints


In [ ]:
if RUN_EXPORT:
    import shutil
    if not (repo_dir / "checkpoints").is_dir():
        raise FileNotFoundError("No checkpoints to export")
    shutil.make_archive(str(repo_dir / "checkpoints"), "zip", repo_dir, "checkpoints")


Published reference: https://www.kaggle.com/code/vhngthnh/qwen-vl-model

The final benchmark and source provenance are under `reports/benchmarks/`.
